In [ ]:
import colour
import numpy as np
import pickle
import pandas as pd
from colour import SpectralShape
import matplotlib.pyplot as plt

import torch
import itertools
lss_path="${repo_root}/assets/lss/just_led.lss"


def display_xy_from_spds(sds):
    xy_values = []
    for sd in sds:
        xy = colour.sd_to_XYZ(sd)
        xy_values.append(colour.XYZ_to_xy(xy))

    #add the cie 1931-2004 chromaticity diagram for reference



    fig, ax = plt.subplots(figsize=(8, 8))

    # Plot the chromaticity diagram (returns the axis)
    colour.plotting.plot_chromaticity_diagram_CIE1931(standalone=False, axes=ax)
    colour.plotting.plot_planckian_locus_in_chromaticity_diagram_CIE1931(standalone=False, axes=ax)
    # Overlay the xy values as red points
    xy_arr = np.array(xy_values)
    ax.scatter(xy_arr[:, 0], xy_arr[:, 1], color='black', label='SPD xy', marker='s')

    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title("xy Chromaticity Coordinates on CIE 1931 Diagram")
    ax.legend()
    ax.grid(True)
    plt.show()



In [ ]:
from scipy.signal import argrelextrema




def read_lss(lss_path):
    """
    Load a numpy array from the given lss file path and convert it to a pandas DataFrame.
    """
    df = pd.read_csv(lss_path, sep="\t", header=0)
    return df
def get_n_maxima(lss_df, n, window=10):
    """
    Get the n spectral distributions (rows) with the highest local maxima,
    ensuring only one SPD per wavelength range (window).
    """
    maxima = []
    for idx, row in lss_df.iterrows():
        values = row.values
        local_maxima = argrelextrema(values, np.greater)[0]
        if len(local_maxima) > 0:
            max_idx = local_maxima[np.argmax(values[local_maxima])]
            max_val = values[max_idx]
            maxima.append((idx, max_idx, max_val))
    # Sort by maximum value descending
    maxima = sorted(maxima, key=lambda x: x[2], reverse=True)
    selected = []
    used_ranges = []
    for idx, max_idx, max_val in maxima:
        # Check if this max_idx is in a new wavelength window
        in_range = any(abs(max_idx - used) < window for used in used_ranges)
        if not in_range:
            selected.append(idx)
            used_ranges.append(max_idx)
        if len(selected) == n:
            break
    return lss_df.iloc[selected]

def df_to_sd(df):
    """
    Convert a DataFrame to a list of SpectralDistribution objects.
    """
    spectral_distributions = []
    wavelengths = df.columns.astype(float)  # Ensure columns are numeric wavelengths
    for index, row in df.iterrows():
        data = row.to_numpy()
        sd = colour.SpectralDistribution(data, wavelengths)
        sd = sd.trim(SpectralShape(299, 800,1))
        spectral_distributions.append(sd)
        
    return spectral_distributions
 
df=read_lss(lss_path)
#using this section to filter from lss file


filtered_df=df

def display_spd(sds):
    print(len(sds))
    for sd in sds:
        plt.plot(sd.wavelengths, sd.values, label=f"SD {sd.wavelengths[0]}-{sd.wavelengths[-1]}")
    plt.xlabel("Wavelength (nm)")
    plt.ylabel("Relative Power")
    plt.title("Spectral Distributions from LSS File")
    # plt.legend()
    plt.grid()
    plt.show()


spectral_distributions=df_to_sd(filtered_df)
#display the spd's using matplotlib
display_spd(spectral_distributions)



In [ ]:
#use colour to convert the spectral distributions to xy

# #display the xy values in matplotlib
# plt.scatter(*zip(*xy_values), label="xy values")
# plt.xlabel("x")
# plt.ylabel("y")
# plt.title("xy Chromaticity Coordinates")
# plt.grid()
# plt.show()

display_xy_from_spds(spectral_distributions)

#cluster=0,1,2


In [ ]:


def df_to_tensor(df):
    """
    Convert a pandas DataFrame to a PyTorch tensor, dropping the first column.
    """
    df_no_first_col = df.iloc[:, 1:]
    return torch.tensor(df_no_first_col.values, dtype=torch.float32)
def convex_interpolation(tensor,s):
    """
    Given:
        tensor: (n, d) tensor, where n is the number of spectral distributions, d is the number of wavelengths.
        n number of samples
    Returns:
        combos: (b, d) tensor, where each row is a weighted combination of the spectral distributions.
        weights: (b, n) tensor, the weights used for each combination.
    """

    
    n, d = tensor.shape
    grid = np.linspace(0, 1, s)
    print(grid)
    # Cartesian product: all possible combinations of weights
    all_weights = np.array(list(itertools.product(grid, repeat=n)))
    weights = torch.tensor(all_weights, dtype=torch.float32)  # (b, n)
    #drop empty rows where all weights are zero
    # weights = weights[~torch.all(weights == 0, dim=1)]  # Remove rows where all weights are zero
    # weights = weights / weights.sum(dim=1, keepdim=True)  # Normalize weights to sum to 1
    #only keep rows where sum of weights is 1
    weights = weights[torch.isclose(weights.sum(dim=1), torch.tensor(1.0), atol=1e-5)] 
    print(weights.shape)


    combos = torch.matmul(weights, tensor)  # (b, d)
    return combos, weights

def dirichlet_interpolation(tensor, alpha, num_samples):
    """
    Given:
        tensor: (n, d) tensor, where n is the number of spectral distributions, d is the number of wavelengths.
        alpha: concentration parameter(s) for the Dirichlet distribution (can be scalar or length-n array).
        num_samples: number of random convex combinations to generate.
    Returns:
        combos: (num_samples, d) tensor, each row is a convex combination of the spectral distributions.
        weights: (num_samples, n) tensor, the Dirichlet weights used for each combination.
    """
    n, d = tensor.shape
    # Sample weights from Dirichlet distribution
    weights_np = np.random.dirichlet(alpha, size=num_samples)  # (num_samples, n)
    weights = torch.tensor(weights_np, dtype=torch.float32)
    combos = torch.matmul(weights, tensor)  # (num_samples, d)
    return combos, weights
tensor = df_to_tensor(filtered_df)

# Perform convex interpolation


convex, convex_weights = convex_interpolation(tensor,6)
alpha = np.array([0.35,0.35,0.35,.8,.8,.8,.8])  # Concentration parameter for Dirichlet distribution
alpha = alpha / alpha.sum()*1.5 # Normalize to ensure it sums to 1
dirichlet, dirichlet_weights = dirichlet_interpolation(tensor, alpha=alpha, num_samples=400)
#set up a dataframe with columns = to  the first df
convex_df = pd.DataFrame(convex.numpy(), columns=filtered_df.columns[1:])  # Exclude the first column
# print(combo_df)
new_spds = df_to_sd(convex_df)
display_spd(new_spds)
display_xy_from_spds(new_spds)

dirichlet_df = pd.DataFrame(dirichlet.numpy(), columns=filtered_df.columns[1:])  # Exclude the first column

new_spds_dirichlet = df_to_sd(dirichlet_df)
display_spd(new_spds_dirichlet)
display_xy_from_spds(new_spds_dirichlet)



In [ ]:
def df_to_lss(df, lss_path):
    """
    Save a pandas DataFrame to a .lss file.
    """
    # Ensure the first column is the wavelength
    df = df.copy()
    df.to_csv(lss_path, sep="\t", index=False, float_format="%.17f")
    print(f"Data saved to {lss_path}")
lss_path = "${repo_root}/assets/lss/conxex.lss"
# Save with float_format to avoid scientific notation
df_to_lss(convex_df, lss_path)

#save the weights as an npy
weights_path = "${repo_root}/assets/lss/convex_weights.npy"
np.save(weights_path, convex_weights.numpy())
# Save the Dirichlet weights
dirichlet_weights_path ="${repo_root}/assets/lss/dirichlet_weights.npy"
dirichlet_lss_path = "${repo_root}/assets/lss/dirichlet.lss"
np.save(dirichlet_weights_path, dirichlet_weights.numpy())
df_to_lss(dirichlet_df, dirichlet_lss_path)



In [ ]:
sanity_check_lss = torch.matmul(torch.tensor([0.5, 0, 0, 0, 0, 0, 0.5], dtype=torch.float32), tensor)
# print(sanity_check_lss)
print(sanity_check_lss.shape)

sanity_df = pd.DataFrame(columns=filtered_df.columns[1:]) 
sanity_df.loc[0] = sanity_check_lss.numpy()
# print(sanity_df)

display_spd(df_to_sd(sanity_df))
path="./edges.lss"
df_to_lss(sanity_df,path)
tele_tensor=[.899953,]



In [ ]:
#additional intution testing

# real_light_path="../python_code/lightbox-data-collector/data/all_SPD_info.p"
# with open(real_light_path, 'rb') as f:
#     real_light_path = pickle.load(f)
# print(len(real_light_path))
real_light_index_path="data/lss/train_val_test_illuminants.p"
# Load the real light source SPD data
with open(real_light_index_path, 'rb') as f:
    real_light_index = pickle.load(f)
train=real_light_index[:250]
val=real_light_index[250:250+60]
test=real_light_index[310:]
# print(train)
# print(val)
# print(test)





In [ ]:
illuminants="${repo_root}/assets/lss/SFU_PhotoLED_LSPDD_RLSS_800W_2022-11-24_10-38-17.lss"
df=read_lss(illuminants)
#parse the numbers out of the string values in each index
# print(real_light_index)
# [print(key)for key in real_light_index]
real_light_ints = [int(key.split('_')[1]) for key in real_light_index]
new_df = df.iloc[real_light_ints].reset_index(drop=True)
# print(len(new_df))
# print(new_df)
#drop the first column (wavelengths) and reset the index
new_df = new_df.drop(columns=new_df.columns[0])
df_to_lss(new_df, "${repo_root}/assets/lss/huang_390_illuminants.lss")

In [ ]:
import numpy as np
import pandas as pd


# Create a DataFrame with the specified columns
led_path="${repo_root}/assets/lss/just_led.lss"
dirchlet_path="${repo_root}/assets/lss/dirichlet.lss"
huang_path="${repo_root}/assets/lss/converted.lss"
temps_path="${repo_root}/assets/lss/temps_1000to10000.lss"

led_df=read_lss(led_path)
print(led_df.shape)
dirchlet_df=read_lss(dirchlet_path)
print(dirchlet_df.shape)
huang_converted=read_lss(huang_path)
print(huang_converted.shape)
temps_df=read_lss(temps_path)
print(temps_df.shape)
#drop temps_df first column and indexes
temps_df = temps_df.drop(columns=temps_df.columns[0])

#drop led.df first column and indexes
led_df = led_df.drop(columns=led_df.columns[0])
#multiply every value in led_df by 3
huang_converted = huang_converted.drop(columns=huang_converted.columns[0])
huang_converted = huang_converted * 3
#combine the two DataFrames
# combined_df = pd.concat([led_df, dirchlet_df,huang_converted,temps_df], ignore_index=True)
combined_df = pd.concat([led_df, dirchlet_df], ignore_index=True)
print(combined_df.shape)
#clip negative values to 0
combined_df = combined_df.clip(lower=0)
# print(combined_df)
# combined_df = combined_df.reset_index(drop=True)
# # Save the combined DataFrame to a new .lss file
combined_lss_path = "${repo_root}/assets/lss/combined.lss"
#add a column at the dataframe named :0.0 with values 5.0
combined_df.insert(0, '0.0', 5.0)
print(combined_df)
# df_to_lss(combined_df, combined_lss_path)





